In [ ]:
!pip install -q elevenlabs

In [ ]:
from IPython.display import Audio, Markdown
from pathlib import Path
from rich.console import Console
from rich.table import Table

def save_audio(path_to_file: Path, audio_generator):
    with path_to_file.open(mode="wb") as file:
        for chunk in audio_generator:
            file.write(chunk)

def print_transcript(transcript):
    print(f"Language: {transcript.language_code} ({transcript.language_probability * 100:.3f}% certainty)")
    display(Markdown(transcript.text))

    console = Console()

    table = Table()
    table.add_column("Text")
    table.add_column("Start")
    table.add_column("End")
    table.add_column("Speaker", style="green")

    for word in transcript.words:
        if word.type == "spacing":
            continue

        table.add_row(word.text, f"{word.start:.3f}", f"{word.end:.3f}", word.speaker_id)

    console.print(table)


def display_audio(path_to_file: Path):
    display(Audio(path_to_file))

In [ ]:
from google.colab import userdata
from elevenlabs import ElevenLabs, play

api_key = userdata.get('ELEVENLABS_API_KEY')
client = ElevenLabs(api_key=api_key)

## Text to Speech

In [ ]:
post_text = """
            🚀 Inception Labs just launched Mercury 2 - the first diffusion-based reasoning LLM. I have high expectations for this approach, and believe that we will be seeing more competitive diffusion-based LLMs emerging in the future.
            Personally, I was fascinated this morning by both the inference speed and the output quality. From my early experiments, the responses were not worse than GPT 5.2 Nano, GPT 5.2 Mini, or Haiku 4.5, but Mercury 2 is blazingly fast. ⚡
            """

In [ ]:
post_audio = client.text_to_speech.convert(
    text=post_text,
    voice_id="nPczCjzI2devNBz1zQrb",
    model_id="eleven_flash_v2_5",
    
    # NOTE: It is not necessary to specify the `output_format` parameter.
    # output_format="mp3_44100_128"
)

PATH_TO_POST_AUDIO = Path("/content/post.mp3")
save_audio(PATH_TO_POST_AUDIO, post_audio)

In [ ]:
display_audio(PATH_TO_POST_AUDIO)

## Speech to Text

The `career_center_softuni.mp3` represents the audio from [this video](https://youtu.be/wMdE2GjBw-A).

The `erlang.mp4` represent [this video](https://youtu.be/M7uo5jmFDUw).

In [ ]:
PATH_TO_CAREER_CENTER_AUDIO = Path("/content/career_center_softuni.mp3")
display_audio(PATH_TO_CAREER_CENTER_AUDIO)

In [ ]:
with Path(PATH_TO_CAREER_CENTER_AUDIO).open(mode="rb") as file:
    career_center_transcript = client.speech_to_text.convert(
        file=file,
        model_id="scribe_v2",
        diarize=True,
        language_code="bg",
        tag_audio_events=True
    )

In [ ]:
print_transcript(career_center_transcript)

In [ ]:
PATH_TO_ERLANG_VIDEO = Path("/content/erlang.mp4")
with PATH_TO_ERLANG_VIDEO.open(mode="rb") as file:
    erlang_transcript = client.speech_to_text.convert(
        file=file,
        model_id="scribe_v2"
    )

In [ ]:
print_transcript(erlang_transcript)

## Speech to Speech

In [ ]:
PATH_TO_TRANSFORMED_CAREER_CENTER_AUDIO = Path("/content/career_center_softuni (transformed).mp3")
with Path(PATH_TO_CAREER_CENTER_AUDIO).open(mode="rb") as file:
    transformed_career_center_audio = client.speech_to_speech.convert(
        audio=file,
        voice_id="M1ydWt7KnBCiuv4CnEDC",
        model_id="eleven_multilingual_sts_v2",
    )

    save_audio(PATH_TO_TRANSFORMED_CAREER_CENTER_AUDIO, transformed_career_center_audio)

In [ ]:
display_audio(PATH_TO_TRANSFORMED_CAREER_CENTER_AUDIO)

In [ ]:
PATH_TO_TRANSFORMED_ERLANG_AUDIO = Path("/content/erlang (transformed).mp3")
with Path(PATH_TO_ERLANG_VIDEO).open(mode="rb") as file:
    transformed_erlang_audio = client.speech_to_speech.convert(
        audio=file,
        voice_id="SOYHLrjzK2X1ezoPC6cr",
        model_id="eleven_english_sts_v2",
    )

    save_audio(PATH_TO_TRANSFORMED_ERLANG_AUDIO, transformed_erlang_audio)

In [ ]:
display_audio(PATH_TO_TRANSFORMED_ERLANG_AUDIO)